total number of features in the original dataset: 166

调整look ahead bias

需要加lag变量：

Close_t

Volume_t

DollarVolume_t

market_cap

r_id

r_cc

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. 读取 stage4_model_data_clean_plus_features.parquet
# ============================================================

possible_paths = [
    Path("stage4_outputs/stage4_model_data_clean_plus_features.parquet"),
    Path("stage4_model_data_clean_plus_features.parquet"),
    Path("outputs/stage4_model_data_clean_plus_features.parquet"),
]

input_path = next((p for p in possible_paths if p.exists()), None)

if input_path is None:
    raise FileNotFoundError(
        "找不到 stage4_model_data_clean_plus_features.parquet，"
        "请检查文件是否在当前目录、stage4_outputs 或 outputs 文件夹里。"
    )

print("Using input file:", input_path)

df = pd.read_parquet(input_path)
print("Original shape:", df.shape)

# ============================================================
# 2. 检查必要列
# ============================================================

if "date" not in df.columns:
    raise ValueError("数据中没有 date 列，无法按时间排序。")

if "instrument_id" in df.columns:
    group_key = "instrument_id"
elif "ticker" in df.columns:
    group_key = "ticker"
else:
    raise ValueError("数据中没有 instrument_id 或 ticker，无法按股票分组做 lag。")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values([group_key, "date"]).copy()

print("Group key:", group_key)

# ============================================================
# 3. 当前值存在 look-ahead risk 的变量
# ============================================================

lookahead_raw_cols = [
    "Close_t",
    "Volume_t",
    "DollarVolume_t",
    "market_cap",
    "r_id",
    "r_cc",
]

existing_lookahead_cols = [c for c in lookahead_raw_cols if c in df.columns]

print("\nFound potential look-ahead columns:")
print(existing_lookahead_cols)

# ============================================================
# 4. 为这些变量生成 lag1 安全版本
# ============================================================
# day t 的 Close_t / Volume_t / market_cap / r_id / r_cc 可能要等收盘后才知道；
# 所以模型在 day t 15:50 时只能使用 t-1 或更早的信息。

for col in existing_lookahead_cols:
    safe_col = f"{col}_lag1_safe"
    df[safe_col] = df.groupby(group_key)[col].shift(1)

print("\nCreated lagged safe versions:")
for col in existing_lookahead_cols:
    print(f"{col}_lag1_safe")

# ============================================================
# 5. 生成更适合模型的 log lag 特征
# ============================================================
# 对价格、成交量、市值这类右偏变量，log 版本通常更适合模型。

if "Close_t_lag1_safe" in df.columns:
    df["log_Close_t_lag1_safe"] = np.log1p(df["Close_t_lag1_safe"].clip(lower=0))

if "Volume_t_lag1_safe" in df.columns:
    df["log_Volume_t_lag1_safe"] = np.log1p(df["Volume_t_lag1_safe"].clip(lower=0))

if "DollarVolume_t_lag1_safe" in df.columns:
    df["log_DollarVolume_t_lag1_safe"] = np.log1p(df["DollarVolume_t_lag1_safe"].clip(lower=0))

if "market_cap_lag1_safe" in df.columns:
    df["log_market_cap_lag1_safe"] = np.log1p(df["market_cap_lag1_safe"].clip(lower=0))

# ============================================================
# 6. 生成横截面 rank 特征
# ============================================================
# 因为你的任务是每天对股票排序，所以 lag 后的变量也可以生成当天横截面排名。

safe_cols_for_rank = [
    "Close_t_lag1_safe",
    "Volume_t_lag1_safe",
    "DollarVolume_t_lag1_safe",
    "market_cap_lag1_safe",
    "r_id_lag1_safe",
    "r_cc_lag1_safe",
    "log_Close_t_lag1_safe",
    "log_Volume_t_lag1_safe",
    "log_DollarVolume_t_lag1_safe",
    "log_market_cap_lag1_safe",
]

safe_cols_for_rank = [c for c in safe_cols_for_rank if c in df.columns]

for col in safe_cols_for_rank:
    df[f"{col}_cs_rank"] = df.groupby("date")[col].rank(pct=True)

print("\nCreated cross-sectional rank features:")
for col in safe_cols_for_rank:
    print(f"{col}_cs_rank")

# ============================================================
# 7. 标记原始 look-ahead 列为不要进模型
# ============================================================
# 注意：这里不一定从数据文件里删除这些列，因为它们可能还可以用于检查或之后计算。
# 但是我们会生成一个 feature exclusion list，训练模型时必须排除它们。

must_exclude_from_X = [
    # target
    "target_r_on_next",
    "target_r_on_next_winsor",

    # ID / split / filter
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "is_trade_eligible",
    "eligibility_reason",

    # 原始 look-ahead risk 变量
    "Close_t",
    "Volume_t",
    "DollarVolume_t",
    "market_cap",
    "r_id",
    "r_cc",
]

# 如果这些变量的 cs_rank / cs_z 已经存在，也必须排除
for base_col in ["Close_t", "Volume_t", "DollarVolume_t", "market_cap", "r_id", "r_cc"]:
    must_exclude_from_X.extend([
        f"{base_col}_cs_rank",
        f"{base_col}_cs_z",
    ])

must_exclude_from_X = [c for c in must_exclude_from_X if c in df.columns]

# ============================================================
# 8. 保存排除列表，方便之后训练模型时使用
# ============================================================

output_dir = Path("stage4_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

exclude_path = output_dir / "lookahead_exclude_from_X.txt"

with open(exclude_path, "w", encoding="utf-8") as f:
    for col in must_exclude_from_X:
        f.write(col + "\n")

print("\nVariables that must be excluded from model X:")
for c in must_exclude_from_X:
    print(c)

print("\nExclusion list saved to:", exclude_path)

# ============================================================
# 9. 生成训练时可用的 feature list
# ============================================================

numeric_cols = [
    c for c in df.columns
    if pd.api.types.is_numeric_dtype(df[c])
]

feature_cols = [
    c for c in numeric_cols
    if c not in must_exclude_from_X
]

feature_list_path = output_dir / "safe_feature_cols_after_lookahead_fix.txt"

with open(feature_list_path, "w", encoding="utf-8") as f:
    for col in feature_cols:
        f.write(col + "\n")

print("\nNumber of safe numeric features:", len(feature_cols))
print("Safe feature list saved to:", feature_list_path)

# ============================================================
# 10. 保存修正后的新数据文件
# ============================================================
# 不覆盖原文件，保存成一个新文件。

output_path = output_dir / "stage4_model_data_clean_plus_features_lookahead_fixed.parquet"

df.to_parquet(output_path, index=False)

print("\nDone.")
print("Fixed file saved to:", output_path)
print("Final shape:", df.shape)

从经济学直观，删除无预测能力以及重复变量
删除以下变量：
target_r_on_next
target_r_on_next_winsor
date
ticker
instrument_id
sample_split
is_trade_eligible
eligibility_reason
impact_bps_at_participation_cap
log_impact_bps
impact_bps 的 cs_rank / cs_z

"adv20",
"adv20_cs_z",
"log_adv20_cs_rank",
"log_adv20_cs_z",

"year_start_market_cap",
"year_start_market_cap_cs_rank",
"year_start_market_cap_cs_z",
"log_market_cap_cs_rank",
"log_market_cap_cs_z",
"market_cap_rank_cs_rank",
"market_cap_rank_cs_z",

"ann_vol60",
"ann_vol60_cs_rank",
"ann_vol60_cs_z",
"vol_60_lag1_cs_z",

"r_on_today_cs_z",
"r_on_lag1_cs_z",
"r_id_lag1_cs_z",
"r_cc_lag1_cs_z",
"r_on_mean_5_cs_z",
"r_on_mean_20_cs_z",
"r_id_mean_5_lag1_cs_z",
"r_id_mean_20_lag1_cs_z",
"r_cc_mean_5_lag1_cs_z",
"r_cc_mean_20_lag1_cs_z",
"abs_r_on_today_cs_z",
"abs_r_cc_lag1_cs_z",
"r_on_today_to_vol20_cs_z",
"r_cc_lag1_to_vol20_cs_z",
"abs_r_on_today_to_vol20_cs_z",
"vol_20_lag1_cs_z",
"vol_60_lag1_cs_z",
"ann_vol60_cs_z",
"price_for_filter_cs_z",
"adv20_cs_z",
"log_adv20_cs_z",
"year_start_market_cap_cs_z",
"log_market_cap_cs_z",
"market_cap_rank_cs_z",
"dsi_cs_z",
"dtcn_cs_z",
"ddtcn_cs_z",
"dsi_lag1_cs_z",
"dtcn_lag1_cs_z",
"ddtcn_lag1_cs_z",
"dsi_change_5_cs_z",
"dsi_change_20_cs_z",
"dtcn_change_5_cs_z",
"dtcn_change_20_cs_z",
"ddtcn_change_5_cs_z",
"ddtcn_change_20_cs_z",
"on_reversal_1d_cs_z",
"id_reversal_1d_cs_z",
"cc_reversal_1d_cs_z",
"on_mom_5_20_cs_z",
"id_mom_5_20_cs_z",
"cc_mom_5_20_cs_z",
"gap_vs_on_mean20_cs_z",
"impact_bps_at_participation_cap_cs_z",
"log_impact_bps_cs_z",

"impact_bps_at_participation_cap",
"log_impact_bps",
"impact_bps_at_participation_cap_cs_rank",
"impact_bps_at_participation_cap_cs_z",
"log_impact_bps_cs_rank",
"log_impact_bps_cs_z",

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. 读取 look-ahead fixed 文件
# ============================================================

possible_paths = [
    Path("stage4_outputs/stage4_model_data_clean_plus_features_lookahead_fixed.parquet"),
    Path("stage4_model_data_clean_plus_features_lookahead_fixed.parquet"),
    Path("outputs/stage4_model_data_clean_plus_features_lookahead_fixed.parquet"),
]

input_path = next((p for p in possible_paths if p.exists()), None)

if input_path is None:
    raise FileNotFoundError(
        "找不到 stage4_model_data_clean_plus_features_lookahead_fixed.parquet，"
        "请检查它是否在当前目录、stage4_outputs 或 outputs 文件夹里。"
    )

print("Using input file:", input_path)

df = pd.read_parquet(input_path)
print("Original shape:", df.shape)


# ============================================================
# 2. 定义 baseline 中需要从 X 排除的变量
# ============================================================

drop_for_baseline = [
    # target：不进入 X，但后面会保留在数据文件中作为 y
    "target_r_on_next",
    "target_r_on_next_winsor",

    # ID / split / filter：不进入 X，但部分会保留用于训练和回测
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "is_trade_eligible",
    "eligibility_reason",

    # cost features：baseline alpha 模型不放
    "impact_bps_at_participation_cap",
    "log_impact_bps",
    "impact_bps_at_participation_cap_cs_rank",
    "impact_bps_at_participation_cap_cs_z",
    "log_impact_bps_cs_rank",
    "log_impact_bps_cs_z",

    # adv20 redundant versions
    "adv20",
    "adv20_cs_z",
    "log_adv20_cs_rank",
    "log_adv20_cs_z",

    # market cap / size redundant versions
    "year_start_market_cap",
    "year_start_market_cap_cs_rank",
    "year_start_market_cap_cs_z",
    "log_market_cap_cs_rank",
    "log_market_cap_cs_z",
    "market_cap_rank_cs_rank",
    "market_cap_rank_cs_z",

    # vol_60 / annual vol redundant versions
    "ann_vol60",
    "ann_vol60_cs_rank",
    "ann_vol60_cs_z",
    "vol_60_lag1_cs_z",

    # all cs_z versions for baseline
    "r_on_today_cs_z",
    "r_on_lag1_cs_z",
    "r_id_lag1_cs_z",
    "r_cc_lag1_cs_z",
    "r_on_mean_5_cs_z",
    "r_on_mean_20_cs_z",
    "r_id_mean_5_lag1_cs_z",
    "r_id_mean_20_lag1_cs_z",
    "r_cc_mean_5_lag1_cs_z",
    "r_cc_mean_20_lag1_cs_z",
    "abs_r_on_today_cs_z",
    "abs_r_cc_lag1_cs_z",
    "r_on_today_to_vol20_cs_z",
    "r_cc_lag1_to_vol20_cs_z",
    "abs_r_on_today_to_vol20_cs_z",
    "vol_20_lag1_cs_z",
    "vol_60_lag1_cs_z",
    "price_for_filter_cs_z",
    "dsi_cs_z",
    "dtcn_cs_z",
    "ddtcn_cs_z",
    "dsi_lag1_cs_z",
    "dtcn_lag1_cs_z",
    "ddtcn_lag1_cs_z",
    "dsi_change_5_cs_z",
    "dsi_change_20_cs_z",
    "dtcn_change_5_cs_z",
    "dtcn_change_20_cs_z",
    "ddtcn_change_5_cs_z",
    "ddtcn_change_20_cs_z",
    "on_reversal_1d_cs_z",
    "id_reversal_1d_cs_z",
    "cc_reversal_1d_cs_z",
    "on_mom_5_20_cs_z",
    "id_mom_5_20_cs_z",
    "cc_mom_5_20_cs_z",
    "gap_vs_on_mean20_cs_z",
]

# 去重，避免同一个变量重复出现
drop_for_baseline = list(dict.fromkeys(drop_for_baseline))

# 只删除数据里真实存在的列
existing_drop_cols = [c for c in drop_for_baseline if c in df.columns]
missing_drop_cols = [c for c in drop_for_baseline if c not in df.columns]

print("\nColumns to exclude from baseline X:", len(existing_drop_cols))
print("Columns in drop list but not found in data:", len(missing_drop_cols))


# ============================================================
# 3. 保留 metadata / target，但不作为 X
# ============================================================
# 这些列保留在删减后的数据文件里，方便后面训练、验证、回测。
# 但它们不会进入 feature_cols。

keep_non_feature_cols = [
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "target_r_on_next",
    "target_r_on_next_winsor",
    "is_trade_eligible",
    "eligibility_reason",
]

keep_non_feature_cols = [c for c in keep_non_feature_cols if c in df.columns]


# ============================================================
# 4. 生成 baseline feature list
# ============================================================

# 从所有列里排除 drop_for_baseline
candidate_cols = [c for c in df.columns if c not in existing_drop_cols]

# 模型 X 只使用数值变量
baseline_feature_cols = [
    c for c in candidate_cols
    if pd.api.types.is_numeric_dtype(df[c])
    and c not in keep_non_feature_cols
]

# 最终删减版数据文件：保留必要 metadata + target + baseline features
final_cols = keep_non_feature_cols + baseline_feature_cols

# 去重保持顺序
final_cols = list(dict.fromkeys(final_cols))

df_reduced = df[final_cols].copy()

print("\nReduced shape:", df_reduced.shape)
print("Number of baseline features:", len(baseline_feature_cols))


# ============================================================
# 5. 保存新文件
# ============================================================

output_dir = Path("stage4_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_data_path = output_dir / "stage4_model_data_baseline_reduced.parquet"
output_feature_list_path = output_dir / "baseline_feature_cols.txt"
output_drop_list_path = output_dir / "baseline_excluded_cols.txt"

df_reduced.to_parquet(output_data_path, index=False)

with open(output_feature_list_path, "w", encoding="utf-8") as f:
    for col in baseline_feature_cols:
        f.write(col + "\n")

with open(output_drop_list_path, "w", encoding="utf-8") as f:
    for col in existing_drop_cols:
        f.write(col + "\n")

print("\nDone.")
print("Reduced data saved to:", output_data_path)
print("Baseline feature list saved to:", output_feature_list_path)
print("Excluded column list saved to:", output_drop_list_path)


# ============================================================
# 6. 打印最终 baseline features，方便检查
# ============================================================

print("\nBaseline feature columns:")
for col in baseline_feature_cols:
    print(col)

特征工程
接下来会逐个完成一下步骤
Step 1：训练 102 feature baseline model
Step 2：按特征类别做 group ablation 
Step 3：用 Boruta 做算法筛选

Step 1：训练 102feature baseline model

这一步是先用全部 102个 baseline 特征训练一个模型。

这一步的目的不是马上筛选，而是建立一个基准线。

可以理解成：

“如果我什么都不再删，就用这 102特征，模型能达到什么效果？”

这个结果非常重要，因为后面所有筛选方法都要和它比较。

比如之后你用 Boruta 只选出了 35 个特征。那你要问：

35 个特征的模型有没有比 102个特征更好？
如果没有更好，至少有没有差不多？
如果差不多，那 35 个特征更简单、更容易解释，也可能更好。

这一步看的是 validation set 表现，而不是 training set 表现。

1. 读取 stage4_model_data_baseline_reduced.parquet
2. 自动识别 110 个特征列
3. 按 sample_split 分成 train （2010-01-04 to 2018-12-31 ） 和 validation（2019-01-02 to 2021-12-31 ）
4. 用 train 训练模型
5. 在 validation 上预测
6. 计算 validation 表现
7. 保存 baseline 模型结果

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib


# =========================
# 1. Set paths
# =========================

DATA_PATH = Path("stage4_model_data_baseline_reduced.parquet")
OUTPUT_DIR = Path("stage4_feature_selection_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# 2. Load data
# =========================

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

print("Data shape:", df.shape)
print("Number of columns:", len(df.columns))

print("\nSample split counts:")
print(df["sample_split"].value_counts(dropna=False))

print("\nSample split date ranges:")
print(
    df.groupby("sample_split")["date"]
    .agg(["min", "max", "count"])
)


# =========================
# 3. Define target
# =========================

target_col = "target_r_on_next_winsor"

if target_col not in df.columns:
    raise ValueError(f"{target_col} is not in the dataframe.")


# =========================
# 4. Identify feature columns
# =========================

non_feature_cols = [
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "target_r_on_next",
    "target_r_on_next_winsor",
    "is_trade_eligible",
    "eligibility_reason"
]

candidate_feature_cols = [
    col for col in df.columns
    if col not in non_feature_cols
]

feature_cols = (
    df[candidate_feature_cols]
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

removed_cols = [
    col for col in candidate_feature_cols
    if col not in feature_cols
]

print("\nNumber of raw columns:", len(df.columns))
print("Number of candidate columns before numeric filtering:", len(candidate_feature_cols))
print("Number of numerical model features used:", len(feature_cols))
print("Non-numeric candidate columns removed:", removed_cols)

print("\nFirst 20 model features:")
print(feature_cols[:20])


# =========================
# 5. Split train and validation
# =========================

train_df = df[df["sample_split"] == "train"].copy()
val_df = df[df["sample_split"] == "valid"].copy()

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain date range:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nValidation date range:")
print(val_df["date"].min(), "to", val_df["date"].max())

if train_df.empty:
    raise ValueError("Train set is empty. Please check sample_split.")

if val_df.empty:
    raise ValueError("Validation set is empty. Your validation split is probably named differently.")


# =========================
# 6. Prepare X and y
# =========================

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]


# =========================
# 7. Handle missing values
# =========================

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)

train_median = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_median)
X_val = X_val.fillna(train_median)


# =========================
# 8. Train baseline model
# =========================

baseline_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=50,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)


# =========================
# 9. Predict on validation set
# =========================

val_pred = baseline_model.predict(X_val)


# =========================
# 10. Calculate validation performance
# =========================

mse = mean_squared_error(y_val, val_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, val_pred)
r2 = r2_score(y_val, val_pred)

val_result = val_df[["date", "ticker", target_col]].copy()
val_result["prediction"] = val_pred

daily_ic = (
    val_result
    .groupby("date")
    .apply(lambda x: x["prediction"].corr(x[target_col], method="spearman"))
    .dropna()
)

mean_daily_ic = daily_ic.mean()


print("\n===== Baseline Validation Performance =====")
print("Number of features used:", len(feature_cols))
print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)
print("Mean daily Spearman IC:", mean_daily_ic)


# =========================
# 11. Save outputs
# =========================

baseline_summary = pd.DataFrame({
    "feature_set": ["baseline_all_numerical_features"],
    "num_features": [len(feature_cols)],
    "train_start": [train_df["date"].min()],
    "train_end": [train_df["date"].max()],
    "valid_start": [val_df["date"].min()],
    "valid_end": [val_df["date"].max()],
    "mse": [mse],
    "rmse": [rmse],
    "mae": [mae],
    "r2": [r2],
    "mean_daily_spearman_ic": [mean_daily_ic]
})

baseline_summary.to_csv(
    OUTPUT_DIR / "step1_baseline_validation_performance.csv",
    index=False
)

val_result.to_csv(
    OUTPUT_DIR / "step1_baseline_validation_predictions.csv",
    index=False
)

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": baseline_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.to_csv(
    OUTPUT_DIR / "step1_baseline_feature_importance.csv",
    index=False
)

pd.Series(feature_cols).to_csv(
    OUTPUT_DIR / "step1_baseline_feature_list.csv",
    index=False,
    header=["feature"]
)

joblib.dump(
    baseline_model,
    OUTPUT_DIR / "step1_baseline_model.pkl"
)

print("\nSaved files:")
print(OUTPUT_DIR / "step1_baseline_validation_performance.csv")
print(OUTPUT_DIR / "step1_baseline_validation_predictions.csv")
print(OUTPUT_DIR / "step1_baseline_feature_importance.csv")
print(OUTPUT_DIR / "step1_baseline_feature_list.csv")
print(OUTPUT_DIR / "step1_baseline_model.pkl")

baseline model 结果
| 指标                          |                      数值 | 含义              |
| --------------------------- | ----------------------: | --------------- |
| `num_features`              |                     102 | 实际进入模型的数值特征数量   |
| `train_start` – `train_end` | 2010-01-04 到 2018-12-31 | 训练集时间           |
| `valid_start` – `valid_end` | 2019-01-02 到 2021-12-31 | 验证集时间           |
| `MSE`                       |                0.000095 | 均方误差            |
| `RMSE`                      |                0.009748 | 预测误差大约是 0.97%   |
| `MAE`                       |                0.006972 | 平均绝对误差大约是 0.70% |
| `R2`                        |               -0.009157 | 解释收益率水平的能力很弱    |
| `Mean daily Spearman IC`    |                0.016659 | 每日横截面排序相关性，正数   |

最重要的十个特征排名：
| 排名 | 特征                    | importance |
| -: | --------------------- | ---------: |
|  1 | `month`               |   0.056651 |
|  2 | `r_on_mean_5`         |   0.053362 |
|  3 | `year`                |   0.048404 |
|  4 | `r_on_lag1`           |   0.045473 |
|  5 | `on_mom_5_20`         |   0.037391 |
|  6 | `day_of_week`         |   0.037265 |
|  7 | `gap_vs_on_mean20`    |   0.031389 |
|  8 | `r_on_today`          |   0.031258 |
|  9 | `on_reversal_1d`      |   0.028912 |
| 10 | `r_on_mean_5_cs_rank` |   0.025225 |

各个大类特征重要性：
| 特征类别                             | 总 importance | 特征数量 | 解释       |
| -------------------------------- | -----------: | ---: | -------- |
| overnight return features        |      约 45.1% |   20 | 最重要      |
| calendar features                |      约 21.1% |   11 | 第二重要     |
| close-to-close features          |      约 16.9% |   14 | 也有一定作用   |
| intraday features                |       约 8.9% |   10 | 中等作用     |
| short-interest / borrow features |       约 5.0% |   36 | 单个特征贡献较弱 |
| volatility features              |       约 2.2% |    4 | 贡献较小     |
| liquidity / size features        |       约 0.9% |    7 | 贡献很小     |


Step 2：按特征类别做 group ablation

Step 2 的目的，是在 Step 1 全部特征 baseline model 的基础上，判断哪一类特征整体对模型最有贡献。具体做法是：先把 102 个模型特征按照经济含义分成几组，比如 overnight return、intraday return、close-to-close return、liquidity/size、volatility、short-interest/borrow、calendar、cross-sectional rank 等；然后以 Step 1 的 baseline validation 表现作为基准，每次有放回地删除其中一组特征，保留其他所有特征，重新训练同一个 Random Forest 模型，并在相同的 validation period 上计算 RMSE、MAE、R² 和最重要的 mean daily Spearman IC；最后把每次删除某一组后的 IC 和 baseline IC 比较，计算 IC_drop = baseline_IC - ablated_IC。如果删除某一组后 IC 明显下降，说明这组特征对模型排序能力很重要；如果删除后 IC 几乎不变，说明这组特征贡献不明显；如果删除后 IC 反而上升，说明这组特征可能带来噪音或过拟合。所以 Step 2 的核心步骤就是：分组 → 删除一组 → 重新训练 → 验证集评估 → 和 baseline 比较 → 判断每类特征的重要性。这一步不是最终决定单个特征，而是先从“特征类别”的层面理解哪些信息来源值得保留，给后面的 Boruta、Elastic Net 和最终 feature set 选择提供依据。

In [12]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import OrderedDict

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ============================================================
# Step 2: Group Ablation Test
# ============================================================

# ------------------------------------------------------------
# 1. 读取数据和 feature list
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

# ============================================================
# 1. 自动寻找 baseline reduced 数据文件
# ============================================================

possible_data_paths = [
    Path("stage4_model_data_baseline_reduced.parquet"),
    Path("stage4_outputs/stage4_model_data_baseline_reduced.parquet"),
    Path("stage4_feature_selection_outputs/stage4_model_data_baseline_reduced.parquet"),
]

data_path = next((p for p in possible_data_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "找不到 stage4_model_data_baseline_reduced.parquet。"
        "请检查它是否在当前目录、stage4_outputs 或 stage4_feature_selection_outputs 里。"
    )

print("Using data file:", data_path)

df = pd.read_parquet(data_path)

print("Data shape:", df.shape)

if not data_path.exists():
    raise FileNotFoundError("找不到 stage4_outputs/stage4_model_data_baseline_reduced.parquet")

if not feature_path.exists():
    raise FileNotFoundError("找不到 stage4_outputs/baseline_feature_cols.txt")

df = pd.read_parquet(data_path)

with open(feature_path, "r", encoding="utf-8") as f:
    feature_cols = [line.strip() for line in f.readlines() if line.strip()]

# 只保留数据中真实存在的特征
feature_cols = [c for c in feature_cols if c in df.columns]

# 只保留数值特征
feature_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

print("Data shape:", df.shape)
print("Number of baseline features:", len(feature_cols))


# ------------------------------------------------------------
# 2. 基本设置
# ------------------------------------------------------------

y_col = "target_r_on_next_winsor"

if y_col not in df.columns:
    raise ValueError(f"数据里找不到目标变量: {y_col}")

if "sample_split" not in df.columns:
    raise ValueError("数据里找不到 sample_split，无法区分 train / validation。")

if "date" not in df.columns:
    raise ValueError("数据里找不到 date，无法计算 daily Rank IC 和 long-short spread。")

df["date"] = pd.to_datetime(df["date"])

# 只使用 eligible stock-days
if "is_trade_eligible" in df.columns:
    df = df[df["is_trade_eligible"] == True].copy()

# 删除 y 缺失的行
df = df.dropna(subset=[y_col]).copy()

split_lower = df["sample_split"].astype(str).str.lower()

train_mask = split_lower.isin(["train", "training"])
valid_mask = split_lower.isin(["valid", "validation", "val"])

train_df = df[train_mask].copy()
valid_df = df[valid_mask].copy()

if len(train_df) == 0:
    raise ValueError("train set 为空，请检查 sample_split 里的标签。")

if len(valid_df) == 0:
    raise ValueError("validation set 为空，请检查 sample_split 里的标签，比如是否叫 valid 或 validation。")

print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))


# ------------------------------------------------------------
# 3. 自动给特征分组
# ------------------------------------------------------------

calendar_names = {
    "month",
    "day_of_week",
    "is_month_end",
    "is_quarter_end",
    "is_monday",
    "is_friday",
    "is_month_start",
    "is_quarter_start",
    "is_year_start",
    "is_year_end",
}

def classify_feature(col):
    c = col.lower()

    # 横截面 rank 单独作为一组
    if c.endswith("_cs_rank"):
        return "cross_sectional_rank"

    # calendar
    if col in calendar_names:
        return "calendar"

    # short-interest
    if (
        c.startswith("dsi")
        or c.startswith("dtcn")
        or c.startswith("ddtcn")
        or "short" in c
    ):
        return "short_interest"

    # volatility / risk
    if (
        c.startswith("vol_")
        or c.startswith("ann_vol")
        or "volatility" in c
        or "sigma" in c
    ):
        return "volatility"

    # liquidity / size / price / volume
    if (
        "adv" in c
        or "market_cap" in c
        or "price" in c
        or "volume" in c
        or "dollarvolume" in c
        or "close_t_lag1_safe" in c
        or "volume_t_lag1_safe" in c
        or "dollarvolume_t_lag1_safe" in c
    ):
        return "liquidity_size"

    # return / momentum / reversal
    if (
        c.startswith("r_on")
        or c.startswith("r_id")
        or c.startswith("r_cc")
        or c.startswith("abs_r")
        or "reversal" in c
        or "mom" in c
        or "gap_vs" in c
    ):
        return "return"

    return "other"


feature_groups = {
    "return": [],
    "volatility": [],
    "liquidity_size": [],
    "short_interest": [],
    "calendar": [],
    "cross_sectional_rank": [],
    "other": [],
}

for col in feature_cols:
    group = classify_feature(col)
    feature_groups[group].append(col)

print("\nFeature group counts:")
for group, cols in feature_groups.items():
    print(f"{group}: {len(cols)}")

if len(feature_groups["other"]) > 0:
    print("\nOther features:")
    for c in feature_groups["other"]:
        print(c)


# ------------------------------------------------------------
# 4. 定义评价指标
# ------------------------------------------------------------

def daily_rank_ic(eval_df, y_col="y", pred_col="pred"):
    """
    每天计算一次 Spearman rank correlation，
    然后取时间序列平均。
    """
    ic_list = []

    for _, g in eval_df.groupby("date"):
        temp = g[[y_col, pred_col]].dropna()

        if len(temp) < 10:
            continue

        if temp[y_col].nunique() <= 1 or temp[pred_col].nunique() <= 1:
            continue

        ic = temp[y_col].corr(temp[pred_col], method="spearman")

        if pd.notna(ic):
            ic_list.append(ic)

    if len(ic_list) == 0:
        return np.nan, np.nan, np.nan, 0

    ic_series = pd.Series(ic_list)

    ic_mean = ic_series.mean()
    ic_std = ic_series.std(ddof=1)

    if pd.notna(ic_std) and ic_std > 0:
        icir = ic_mean / ic_std * np.sqrt(252)
    else:
        icir = np.nan

    return ic_mean, ic_std, icir, len(ic_series)


def daily_top_bottom_spread(eval_df, y_col="y", pred_col="pred", q=0.2):
    """
    每天按预测值排序：
    做多 top q，做空 bottom q。
    返回每日 top-bottom realized return 的均值和 Sharpe。
    """
    spread_list = []

    for _, g in eval_df.groupby("date"):
        temp = g[[y_col, pred_col]].dropna()

        if len(temp) < 20:
            continue

        temp = temp.sort_values(pred_col)

        k = max(int(len(temp) * q), 1)

        bottom = temp.head(k)[y_col].mean()
        top = temp.tail(k)[y_col].mean()

        spread = top - bottom
        spread_list.append(spread)

    if len(spread_list) == 0:
        return np.nan, np.nan, np.nan, 0

    spread_series = pd.Series(spread_list)

    mean_spread = spread_series.mean()
    std_spread = spread_series.std(ddof=1)

    if pd.notna(std_spread) and std_spread > 0:
        sharpe = mean_spread / std_spread * np.sqrt(252)
    else:
        sharpe = np.nan

    mean_spread_bps = mean_spread * 10000

    return mean_spread, mean_spread_bps, sharpe, len(spread_series)


# ------------------------------------------------------------
# 5. 定义模型
# ------------------------------------------------------------
# 这里用 Ridge 做 group ablation，因为它速度快、稳定、容易比较。
# 后面正式模型可以换成 Random Forest / Gradient Boosting / XGBoost / LightGBM。

def make_model():
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])
    return model


# ------------------------------------------------------------
# 6. 构造不同的 group ablation 实验
# ------------------------------------------------------------

def unique_keep_order(cols):
    return list(dict.fromkeys(cols))


experiments = OrderedDict()

return_cols = feature_groups["return"]
vol_cols = feature_groups["volatility"]
liq_cols = feature_groups["liquidity_size"]
short_cols = feature_groups["short_interest"]
cal_cols = feature_groups["calendar"]
rank_cols = feature_groups["cross_sectional_rank"]
other_cols = feature_groups["other"]

experiments["01_return_only"] = unique_keep_order(
    return_cols
)

experiments["02_return_plus_volatility"] = unique_keep_order(
    return_cols + vol_cols
)

experiments["03_return_vol_liquidity_size"] = unique_keep_order(
    return_cols + vol_cols + liq_cols
)

experiments["04_return_vol_liq_short_interest"] = unique_keep_order(
    return_cols + vol_cols + liq_cols + short_cols
)

experiments["05_return_vol_liq_short_calendar"] = unique_keep_order(
    return_cols + vol_cols + liq_cols + short_cols + cal_cols
)

experiments["06_core_plus_cross_sectional_rank"] = unique_keep_order(
    return_cols + vol_cols + liq_cols + short_cols + cal_cols + rank_cols
)

experiments["07_cross_sectional_rank_only"] = unique_keep_order(
    rank_cols
)

experiments["08_all_baseline_features"] = unique_keep_order(
    feature_cols
)

# other 特征存在时，额外测试一次
if len(other_cols) > 0:
    experiments["09_all_plus_other_check"] = unique_keep_order(
        return_cols + vol_cols + liq_cols + short_cols + cal_cols + rank_cols + other_cols
    )


# ------------------------------------------------------------
# 7. 逐个实验训练模型并评价 validation 表现
# ------------------------------------------------------------

results = []
predictions_by_experiment = {}

for exp_name, cols in experiments.items():
    cols = [c for c in cols if c in feature_cols]

    if len(cols) == 0:
        print(f"Skip {exp_name}: no features.")
        continue

    print(f"\nRunning {exp_name} with {len(cols)} features...")

    X_train = train_df[cols]
    y_train = train_df[y_col]

    X_valid = valid_df[cols]
    y_valid = valid_df[y_col]

    model = make_model()
    model.fit(X_train, y_train)

    pred_valid = model.predict(X_valid)

    eval_valid = pd.DataFrame({
        "date": valid_df["date"].values,
        "y": y_valid.values,
        "pred": pred_valid,
    })

    ic_mean, ic_std, icir, n_ic_days = daily_rank_ic(
        eval_valid,
        y_col="y",
        pred_col="pred"
    )

    spread_mean, spread_bps, spread_sharpe, n_spread_days = daily_top_bottom_spread(
        eval_valid,
        y_col="y",
        pred_col="pred",
        q=0.2
    )

    mse = mean_squared_error(y_valid, pred_valid)
    mae = mean_absolute_error(y_valid, pred_valid)

    results.append({
        "experiment": exp_name,
        "n_features": len(cols),
        "valid_rank_ic_mean": ic_mean,
        "valid_rank_ic_std": ic_std,
        "valid_icir_annualized": icir,
        "valid_top_bottom_mean": spread_mean,
        "valid_top_bottom_bps": spread_bps,
        "valid_top_bottom_sharpe": spread_sharpe,
        "valid_mse": mse,
        "valid_mae": mae,
        "n_ic_days": n_ic_days,
        "n_spread_days": n_spread_days,
    })

    predictions_by_experiment[exp_name] = eval_valid


# ------------------------------------------------------------
# 8. 整理结果
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

# 排序标准：优先看 Rank IC，其次看 top-bottom Sharpe
results_df = results_df.sort_values(
    ["valid_rank_ic_mean", "valid_top_bottom_sharpe"],
    ascending=False
).reset_index(drop=True)

print("\nGroup ablation results:")
display(results_df)


# ------------------------------------------------------------
# 9. 保存结果
# ------------------------------------------------------------

output_dir = Path("stage4_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

result_path = output_dir / "step2_group_ablation_results.csv"
group_path = output_dir / "step2_feature_groups.json"
best_feature_path = output_dir / "step2_best_group_ablation_features.txt"

results_df.to_csv(result_path, index=False)

# 保存每个 group 里面有哪些特征
with open(group_path, "w", encoding="utf-8") as f:
    json.dump(feature_groups, f, indent=2)

# 保存 validation 表现最好的特征组合
best_experiment = results_df.loc[0, "experiment"]
best_features = experiments[best_experiment]

with open(best_feature_path, "w", encoding="utf-8") as f:
    for col in best_features:
        f.write(col + "\n")

print("\nDone.")
print("Best experiment:", best_experiment)
print("Best number of features:", len(best_features))
print("Results saved to:", result_path)
print("Feature groups saved to:", group_path)
print("Best feature list saved to:", best_feature_path)

Using data file: stage4_model_data_baseline_reduced.parquet
Data shape: (3773971, 110)
Data shape: (3773971, 110)
Number of baseline features: 102
Train rows: 1758771
Valid rows: 702218

Feature group counts:
return: 22
volatility: 2
liquidity_size: 5
short_interest: 24
calendar: 10
cross_sectional_rank: 38
other: 1

Other features:
year

Running 01_return_only with 22 features...

Running 02_return_plus_volatility with 24 features...

Running 03_return_vol_liquidity_size with 29 features...

Running 04_return_vol_liq_short_interest with 53 features...

Running 05_return_vol_liq_short_calendar with 63 features...

Running 06_core_plus_cross_sectional_rank with 101 features...

Running 07_cross_sectional_rank_only with 38 features...

Running 08_all_baseline_features with 102 features...

Running 09_all_plus_other_check with 102 features...

Group ablation results:


,experiment,n_features,valid_rank_ic_mean,valid_rank_ic_std,valid_icir_annualized,valid_top_bottom_mean,valid_top_bottom_bps,valid_top_bottom_sharpe,valid_mse,valid_mae,n_ic_days,n_spread_days
0,03_return_vol_liquidity_size,29,0.031097,0.229333,2.152541,0.000372,3.721009,1.439241,0.000089,0.006758,757,757
1,04_return_vol_liq_short_interest,53,0.028863,0.223311,2.051777,0.000374,3.736862,1.483047,0.000089,0.006760,757,757
2,05_return_vol_liq_short_calendar,63,0.028540,0.223006,2.031630,0.000368,3.675572,1.457932,0.000089,0.006765,757,757
3,06_core_plus_cross_sectional_rank,101,0.027685,0.178993,2.455331,0.000387,3.874591,1.837727,0.000090,0.006793,757,757
4,08_all_baseline_features,102,0.027568,0.177974,2.458907,0.000382,3.816424,1.817116,0.000090,0.006789,757,757
5,09_all_plus_other_check,102,0.027568,0.177974,2.458907,0.000382,3.816424,1.817116,0.000090,0.006789,757,757
6,01_return_only,22,0.026958,0.237798,1.799609,0.000332,3.319747,1.272146,0.000089,0.006766,757,757
7,02_return_plus_volatility,24,0.026439,0.231415,1.813645,0.000336,3.359883,1.316767,0.000089,0.006766,757,757
8,07_cross_sectional_rank_only,38,0.020309,0.217586,1.481684,0.000303,3.031230,1.261632,0.000089,0.006773,757,757



Done.
Best experiment: 03_return_vol_liquidity_size
Best number of features: 29
Results saved to: stage4_outputs\step2_group_ablation_results.csv
Feature groups saved to: stage4_outputs\step2_feature_groups.json
Best feature list saved to: stage4_outputs\step2_best_group_ablation_features.txt


step 2 结论：

将102个特征分成7类：| 特征组                    | 数量 | 含义                     |
| ---------------------- | -: | ---------------------- |
| `return`               | 22 | 收益率、动量、反转类特征           |
| `volatility`           |  2 | 波动率特征                  |
| `liquidity_size`       |  5 | 流动性、市值、价格相关特征          |
| `short_interest`       | 24 | 卖空压力、days-to-cover 等特征 |
| `calendar`             | 10 | 周几、月末、季末等特征            |
| `cross_sectional_rank` | 38 | 每天横截面排名特征              |
| `other`                |  1 | `year`                 |

Return features 是基础 alpha 信号，只用 return 特征已经有正的 Rank IC 和 long-short spread。

Liquidity / size 是提升 Rank IC 最明显的特征组，加入后 Rank IC 达到全表最高的 0.031097。


Calendar features 没有明显帮助，加入后 Rank IC、spread 和 Sharpe 都略微下降。

Cross-sectional rank 特征对最终交易策略最有用，让 top-bottom bps 和 Sharpe 达到全表最高。

只用 cross-sectional rank 不够，它需要和原始经济特征结合使用。

全部 102 个 baseline 特征并不是最优，说明继续做算法筛选是有必要的。


step 3: Boruta feature selection

Step 3 的目的，是在你已经通过 Step 2 找到“表现较好的特征组”之后，进一步用 Boruta 判断这些特征里哪些是真的有预测力、哪些可能只是噪声。具体可以分成 5 小步：第一步，确定输入特征集,二步，只在 train set 上运行 Boruta，不使用 validation 和 test，避免特征筛选过程偷看未来样本；第三步，Boruta 会为每个真实特征生成一个打乱顺序的 shadow feature，然后用 Random Forest 同时训练真实特征和 shadow features；第四步，Boruta 比较真实特征的重要性是否稳定超过 shadow features，如果超过就标记为 confirmed，不明显就标记为 tentative，明显没用就标记为 rejected；第五步，用 Boruta 选出的特征重新训练模型，并在 validation set 上和 Step 2 的最佳特征组、完整 102-feature baseline 做比较。简单说，Step 3 不是直接相信 Boruta，而是让 Boruta给你一个“算法筛选后的候选特征集”，然后再用 validation 表现决定它是否真的比原来的特征组更好。

In [14]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import binomtest

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ============================================================
# Step 3B: Boruta on 06_core_plus_cross_sectional_rank
# ============================================================

RANDOM_STATE = 2026

# 如果运行太慢，可以先改小，例如 50_000
MAX_BORUTA_ROWS = 120_000

# 如果运行太慢，可以先改成 10；正式结果建议 20
N_BORUTA_ITER = 20

# Random Forest 参数
RF_N_ESTIMATORS = 150
RF_MAX_DEPTH = 7
RF_MIN_SAMPLES_LEAF = 50

# Boruta 显著性水平
ALPHA = 0.05


# ============================================================
# 1. 读取 baseline reduced 数据
# ============================================================

possible_data_paths = [
    Path("stage4_model_data_baseline_reduced.parquet"),
    Path("stage4_outputs/stage4_model_data_baseline_reduced.parquet"),
    Path("stage4_feature_selection_outputs/stage4_model_data_baseline_reduced.parquet"),
]

data_path = next((p for p in possible_data_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "找不到 stage4_model_data_baseline_reduced.parquet。"
        "请检查它是否在当前目录、stage4_outputs 或 stage4_feature_selection_outputs 里。"
    )

print("Using data file:", data_path)

df = pd.read_parquet(data_path)

if "date" not in df.columns:
    raise ValueError("数据里没有 date 列。")

if "sample_split" not in df.columns:
    raise ValueError("数据里没有 sample_split 列。")

y_col = "target_r_on_next_winsor"

if y_col not in df.columns:
    raise ValueError(f"数据里没有目标变量 {y_col}。")

df["date"] = pd.to_datetime(df["date"])

print("Data shape:", df.shape)


# ============================================================
# 2. 自动生成全部 baseline numeric features
# ============================================================

non_feature_cols = [
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "target_r_on_next",
    "target_r_on_next_winsor",
    "is_trade_eligible",
    "eligibility_reason",
]

all_numeric_features = [
    c for c in df.columns
    if c not in non_feature_cols
    and pd.api.types.is_numeric_dtype(df[c])
]

print("Total numeric baseline features:", len(all_numeric_features))


# ============================================================
# 3. 按 Step 2 逻辑重新给特征分组
# ============================================================

calendar_names = {
    "month",
    "day_of_week",
    "is_month_end",
    "is_quarter_end",
    "is_monday",
    "is_friday",
    "is_month_start",
    "is_quarter_start",
    "is_year_start",
    "is_year_end",
}

def classify_feature(col):
    c = col.lower()

    # 横截面 rank 单独作为一组
    if c.endswith("_cs_rank"):
        return "cross_sectional_rank"

    # calendar
    if col in calendar_names:
        return "calendar"

    # short-interest
    if (
        c.startswith("dsi")
        or c.startswith("dtcn")
        or c.startswith("ddtcn")
        or "short" in c
    ):
        return "short_interest"

    # volatility / risk
    if (
        c.startswith("vol_")
        or c.startswith("ann_vol")
        or "volatility" in c
        or "sigma" in c
    ):
        return "volatility"

    # liquidity / size / price / volume
    if (
        "adv" in c
        or "market_cap" in c
        or "price" in c
        or "volume" in c
        or "dollarvolume" in c
        or "close_t_lag1_safe" in c
        or "volume_t_lag1_safe" in c
        or "dollarvolume_t_lag1_safe" in c
    ):
        return "liquidity_size"

    # return / momentum / reversal
    if (
        c.startswith("r_on")
        or c.startswith("r_id")
        or c.startswith("r_cc")
        or c.startswith("abs_r")
        or "reversal" in c
        or "mom" in c
        or "gap_vs" in c
    ):
        return "return"

    return "other"


feature_groups = {
    "return": [],
    "volatility": [],
    "liquidity_size": [],
    "short_interest": [],
    "calendar": [],
    "cross_sectional_rank": [],
    "other": [],
}

for col in all_numeric_features:
    group = classify_feature(col)
    feature_groups[group].append(col)

print("\nFeature group counts:")
for group, cols in feature_groups.items():
    print(f"{group}: {len(cols)}")

if len(feature_groups["other"]) > 0:
    print("\nOther features:")
    for c in feature_groups["other"]:
        print(c)


# ============================================================
# 4. 构造 06_core_plus_cross_sectional_rank 特征组
# ============================================================
# 这一步是关键：不用 step2_best_group_ablation_features.txt，
# 而是明确构造 Step 2 中 Sharpe 最高的 06 组。

candidate_features = (
    feature_groups["return"]
    + feature_groups["volatility"]
    + feature_groups["liquidity_size"]
    + feature_groups["short_interest"]
    + feature_groups["calendar"]
    + feature_groups["cross_sectional_rank"]
)

# 去重并保持顺序
candidate_features = list(dict.fromkeys(candidate_features))

# 再确认都在 df 里，并且是数值变量
candidate_features = [
    c for c in candidate_features
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]

print("\nCandidate feature set: 06_core_plus_cross_sectional_rank")
print("Number of candidate features:", len(candidate_features))

if len(candidate_features) == 0:
    raise ValueError("candidate_features 为空，无法运行 Boruta。")

print("\nFirst 30 candidate features:")
for c in candidate_features[:30]:
    print(c)


# ============================================================
# 5. 准备 train / validation 数据
# ============================================================

df = df.dropna(subset=[y_col]).copy()

if "is_trade_eligible" in df.columns:
    df = df[df["is_trade_eligible"] == True].copy()

split_lower = df["sample_split"].astype(str).str.lower()

train_mask = split_lower.isin(["train", "training"])
valid_mask = split_lower.isin(["valid", "validation", "val"])

train_df = df[train_mask].copy()
valid_df = df[valid_mask].copy()

print("\nTrain rows:", len(train_df))
print("Valid rows:", len(valid_df))

if len(train_df) == 0:
    raise ValueError("train set 为空，请检查 sample_split。")

if len(valid_df) == 0:
    raise ValueError("validation set 为空，请检查 sample_split。")


# ============================================================
# 6. 为 Boruta 抽样 train set
# ============================================================

if len(train_df) > MAX_BORUTA_ROWS:
    boruta_train_df = train_df.sample(
        n=MAX_BORUTA_ROWS,
        random_state=RANDOM_STATE
    ).copy()
else:
    boruta_train_df = train_df.copy()

print("Rows used for Boruta:", len(boruta_train_df))


# ============================================================
# 7. 缺失值填补
# ============================================================

imputer = SimpleImputer(strategy="median")

X_boruta = imputer.fit_transform(boruta_train_df[candidate_features])
y_boruta = boruta_train_df[y_col].values

X_boruta = X_boruta.astype(np.float32)

print("Boruta X shape:", X_boruta.shape)


# ============================================================
# 8. Boruta-style shadow feature selection
# ============================================================

n_features = len(candidate_features)

hit_counts = np.zeros(n_features, dtype=int)
importance_history = []
shadow_max_history = []

rng = np.random.default_rng(RANDOM_STATE)

for it in range(N_BORUTA_ITER):
    print(f"\nBoruta iteration {it + 1}/{N_BORUTA_ITER}")

    # 生成 shadow features：每个真实特征打乱一份
    X_shadow = X_boruta.copy()

    for j in range(n_features):
        rng.shuffle(X_shadow[:, j])

    X_aug = np.hstack([X_boruta, X_shadow])

    rf = RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features="sqrt",
        random_state=RANDOM_STATE + it,
        n_jobs=-1,
    )

    rf.fit(X_aug, y_boruta)

    importances = rf.feature_importances_

    real_importance = importances[:n_features]
    shadow_importance = importances[n_features:]

    shadow_max = np.max(shadow_importance)

    hits = real_importance > shadow_max
    hit_counts += hits.astype(int)

    importance_history.append(real_importance)
    shadow_max_history.append(shadow_max)

    print("Shadow max importance:", shadow_max)
    print("Number of real features beating shadow max:", hits.sum())


importance_history = np.vstack(importance_history)
mean_importance = importance_history.mean(axis=0)
std_importance = importance_history.std(axis=0)


# ============================================================
# 9. 根据 hit count 判定 confirmed / tentative / rejected
# ============================================================

decisions = []

for hits in hit_counts:
    p_greater = binomtest(
        hits,
        N_BORUTA_ITER,
        0.5,
        alternative="greater"
    ).pvalue

    p_less = binomtest(
        hits,
        N_BORUTA_ITER,
        0.5,
        alternative="less"
    ).pvalue

    if p_greater < ALPHA and hits > N_BORUTA_ITER / 2:
        decision = "confirmed"
    elif p_less < ALPHA and hits < N_BORUTA_ITER / 2:
        decision = "rejected"
    else:
        decision = "tentative"

    decisions.append(decision)


boruta_report = pd.DataFrame({
    "feature": candidate_features,
    "decision": decisions,
    "hit_count": hit_counts,
    "n_iter": N_BORUTA_ITER,
    "hit_rate": hit_counts / N_BORUTA_ITER,
    "mean_importance": mean_importance,
    "std_importance": std_importance,
})

decision_order = {
    "confirmed": 0,
    "tentative": 1,
    "rejected": 2,
}

boruta_report["decision_order"] = boruta_report["decision"].map(decision_order)

boruta_report = boruta_report.sort_values(
    ["decision_order", "hit_rate", "mean_importance"],
    ascending=[True, False, False]
).drop(columns=["decision_order"]).reset_index(drop=True)

print("\nBoruta decision counts:")
print(boruta_report["decision"].value_counts())

print("\nTop Boruta features:")
try:
    display(boruta_report.head(40))
except NameError:
    print(boruta_report.head(40).to_string(index=False))


confirmed_features = boruta_report.loc[
    boruta_report["decision"] == "confirmed",
    "feature"
].tolist()

tentative_features = boruta_report.loc[
    boruta_report["decision"] == "tentative",
    "feature"
].tolist()

rejected_features = boruta_report.loc[
    boruta_report["decision"] == "rejected",
    "feature"
].tolist()

confirmed_plus_tentative = confirmed_features + tentative_features

print("\nNumber of confirmed features:", len(confirmed_features))
print("Number of tentative features:", len(tentative_features))
print("Number of rejected features:", len(rejected_features))


# ============================================================
# 10. 定义 validation 评价函数
# ============================================================

def daily_rank_ic(eval_df, y_col="y", pred_col="pred"):
    ic_list = []

    for _, g in eval_df.groupby("date"):
        temp = g[[y_col, pred_col]].dropna()

        if len(temp) < 10:
            continue

        if temp[y_col].nunique() <= 1 or temp[pred_col].nunique() <= 1:
            continue

        ic = temp[y_col].corr(temp[pred_col], method="spearman")

        if pd.notna(ic):
            ic_list.append(ic)

    if len(ic_list) == 0:
        return np.nan, np.nan, np.nan, 0

    ic_series = pd.Series(ic_list)

    ic_mean = ic_series.mean()
    ic_std = ic_series.std(ddof=1)

    if pd.notna(ic_std) and ic_std > 0:
        icir = ic_mean / ic_std * np.sqrt(252)
    else:
        icir = np.nan

    return ic_mean, ic_std, icir, len(ic_series)


def daily_top_bottom_spread(eval_df, y_col="y", pred_col="pred", q=0.2):
    spread_list = []

    for _, g in eval_df.groupby("date"):
        temp = g[[y_col, pred_col]].dropna()

        if len(temp) < 20:
            continue

        temp = temp.sort_values(pred_col)

        k = max(int(len(temp) * q), 1)

        bottom = temp.head(k)[y_col].mean()
        top = temp.tail(k)[y_col].mean()

        spread = top - bottom
        spread_list.append(spread)

    if len(spread_list) == 0:
        return np.nan, np.nan, np.nan, 0

    spread_series = pd.Series(spread_list)

    mean_spread = spread_series.mean()
    std_spread = spread_series.std(ddof=1)

    if pd.notna(std_spread) and std_spread > 0:
        sharpe = mean_spread / std_spread * np.sqrt(252)
    else:
        sharpe = np.nan

    mean_spread_bps = mean_spread * 10000

    return mean_spread, mean_spread_bps, sharpe, len(spread_series)


# ============================================================
# 11. 用 Ridge 比较不同 Boruta 特征集的 validation 表现
# ============================================================

def make_ridge_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ])


def evaluate_feature_set(feature_set_name, cols):
    cols = [c for c in cols if c in df.columns]

    if len(cols) == 0:
        print(f"Skip {feature_set_name}: no features.")
        return None

    print(f"\nEvaluating {feature_set_name} with {len(cols)} features...")

    X_train = train_df[cols]
    y_train = train_df[y_col]

    X_valid = valid_df[cols]
    y_valid = valid_df[y_col]

    model = make_ridge_model()
    model.fit(X_train, y_train)

    pred_valid = model.predict(X_valid)

    eval_valid = pd.DataFrame({
        "date": valid_df["date"].values,
        "y": y_valid.values,
        "pred": pred_valid,
    })

    ic_mean, ic_std, icir, n_ic_days = daily_rank_ic(
        eval_valid,
        y_col="y",
        pred_col="pred"
    )

    spread_mean, spread_bps, spread_sharpe, n_spread_days = daily_top_bottom_spread(
        eval_valid,
        y_col="y",
        pred_col="pred",
        q=0.2
    )

    mse = mean_squared_error(y_valid, pred_valid)
    mae = mean_absolute_error(y_valid, pred_valid)

    return {
        "feature_set": feature_set_name,
        "n_features": len(cols),
        "valid_rank_ic_mean": ic_mean,
        "valid_rank_ic_std": ic_std,
        "valid_icir_annualized": icir,
        "valid_top_bottom_mean": spread_mean,
        "valid_top_bottom_bps": spread_bps,
        "valid_top_bottom_sharpe": spread_sharpe,
        "valid_mse": mse,
        "valid_mae": mae,
        "n_ic_days": n_ic_days,
        "n_spread_days": n_spread_days,
    }


comparison_results = []

# 1. Step 2 Sharpe 最高的 06 原始特征组
res = evaluate_feature_set("06_core_plus_cross_sectional_rank", candidate_features)
if res is not None:
    comparison_results.append(res)

# 2. Boruta confirmed only
if len(confirmed_features) > 0:
    res = evaluate_feature_set("boruta_06_confirmed", confirmed_features)
    if res is not None:
        comparison_results.append(res)

# 3. Boruta confirmed + tentative
if len(confirmed_plus_tentative) > 0:
    res = evaluate_feature_set("boruta_06_confirmed_plus_tentative", confirmed_plus_tentative)
    if res is not None:
        comparison_results.append(res)

# 4. Top 30 by importance
top30_features = boruta_report.sort_values(
    "mean_importance",
    ascending=False
)["feature"].head(30).tolist()

res = evaluate_feature_set("boruta_06_top30_by_importance", top30_features)
if res is not None:
    comparison_results.append(res)

# 5. Top 50 by importance
top50_features = boruta_report.sort_values(
    "mean_importance",
    ascending=False
)["feature"].head(50).tolist()

res = evaluate_feature_set("boruta_06_top50_by_importance", top50_features)
if res is not None:
    comparison_results.append(res)

# 6. Top 70 by importance
top70_features = boruta_report.sort_values(
    "mean_importance",
    ascending=False
)["feature"].head(70).tolist()

res = evaluate_feature_set("boruta_06_top70_by_importance", top70_features)
if res is not None:
    comparison_results.append(res)


comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    ["valid_top_bottom_sharpe", "valid_rank_ic_mean"],
    ascending=False
).reset_index(drop=True)

print("\nStep 3B Boruta validation comparison for 06_core_plus_cross_sectional_rank:")
try:
    display(comparison_df)
except NameError:
    print(comparison_df.to_string(index=False))


# ============================================================
# 12. 保存 Step 3B 输出结果
# ============================================================

output_dir = Path("stage4_feature_selection_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

boruta_report_path = output_dir / "step3b_boruta_06_feature_report.csv"
comparison_path = output_dir / "step3b_boruta_06_validation_comparison.csv"

candidate_path = output_dir / "step3b_06_candidate_features.txt"
confirmed_path = output_dir / "step3b_boruta_06_confirmed_features.txt"
tentative_path = output_dir / "step3b_boruta_06_tentative_features.txt"
rejected_path = output_dir / "step3b_boruta_06_rejected_features.txt"
confirmed_tentative_path = output_dir / "step3b_boruta_06_confirmed_plus_tentative_features.txt"

top30_path = output_dir / "step3b_boruta_06_top30_features.txt"
top50_path = output_dir / "step3b_boruta_06_top50_features.txt"
top70_path = output_dir / "step3b_boruta_06_top70_features.txt"

boruta_report.to_csv(boruta_report_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

with open(candidate_path, "w", encoding="utf-8") as f:
    for col in candidate_features:
        f.write(col + "\n")

with open(confirmed_path, "w", encoding="utf-8") as f:
    for col in confirmed_features:
        f.write(col + "\n")

with open(tentative_path, "w", encoding="utf-8") as f:
    for col in tentative_features:
        f.write(col + "\n")

with open(rejected_path, "w", encoding="utf-8") as f:
    for col in rejected_features:
        f.write(col + "\n")

with open(confirmed_tentative_path, "w", encoding="utf-8") as f:
    for col in confirmed_plus_tentative:
        f.write(col + "\n")

with open(top30_path, "w", encoding="utf-8") as f:
    for col in top30_features:
        f.write(col + "\n")

with open(top50_path, "w", encoding="utf-8") as f:
    for col in top50_features:
        f.write(col + "\n")

with open(top70_path, "w", encoding="utf-8") as f:
    for col in top70_features:
        f.write(col + "\n")


# ============================================================
# 13. 自动保存当前 Step 3B 最佳特征集
# ============================================================

best_feature_set_name = comparison_df.loc[0, "feature_set"]

if best_feature_set_name == "06_core_plus_cross_sectional_rank":
    best_features = candidate_features
elif best_feature_set_name == "boruta_06_confirmed":
    best_features = confirmed_features
elif best_feature_set_name == "boruta_06_confirmed_plus_tentative":
    best_features = confirmed_plus_tentative
elif best_feature_set_name == "boruta_06_top30_by_importance":
    best_features = top30_features
elif best_feature_set_name == "boruta_06_top50_by_importance":
    best_features = top50_features
elif best_feature_set_name == "boruta_06_top70_by_importance":
    best_features = top70_features
else:
    best_features = candidate_features

best_path = output_dir / "step3b_best_boruta_06_feature_set.txt"

with open(best_path, "w", encoding="utf-8") as f:
    for col in best_features:
        f.write(col + "\n")


print("\nDone.")
print("06 candidate feature list saved to:", candidate_path)
print("Boruta feature report saved to:", boruta_report_path)
print("Validation comparison saved to:", comparison_path)
print("Confirmed features saved to:", confirmed_path)
print("Tentative features saved to:", tentative_path)
print("Rejected features saved to:", rejected_path)
print("Best Step 3B feature set:", best_feature_set_name)
print("Best Step 3B feature list saved to:", best_path)

Using data file: stage4_model_data_baseline_reduced.parquet
Data shape: (3773971, 110)
Total numeric baseline features: 102

Feature group counts:
return: 22
volatility: 2
liquidity_size: 5
short_interest: 24
calendar: 10
cross_sectional_rank: 38
other: 1

Other features:
year

Candidate feature set: 06_core_plus_cross_sectional_rank
Number of candidate features: 101

First 30 candidate features:
r_on_today
r_on_lag1
r_id_lag1
r_cc_lag1
r_on_mean_5
r_on_mean_20
r_id_mean_5_lag1
r_id_mean_20_lag1
r_cc_mean_5_lag1
r_cc_mean_20_lag1
abs_r_on_today
abs_r_cc_lag1
on_reversal_1d
id_reversal_1d
cc_reversal_1d
on_mom_5_20
gap_vs_on_mean20
id_mom_5_20
cc_mom_5_20
r_on_today_to_vol20
r_cc_lag1_to_vol20
abs_r_on_today_to_vol20
vol_20_lag1
vol_60_lag1
price_for_filter
market_cap_rank
log_adv20
log_market_cap
log_price_for_filter
dsi

Train rows: 1758771
Valid rows: 702218
Rows used for Boruta: 120000
Boruta X shape: (120000, 101)

Boruta iteration 1/20
Shadow max importance: 0.003055911403777174
N

,feature,decision,hit_count,n_iter,hit_rate,mean_importance,std_importance
0,r_on_mean_5,confirmed,20,20,1.0,0.054743,0.003346
1,on_mom_5_20,confirmed,20,20,1.0,0.046255,0.003090
2,r_on_lag1,confirmed,20,20,1.0,0.037990,0.002733
3,gap_vs_on_mean20,confirmed,20,20,1.0,0.033097,0.002552
4,on_reversal_1d,confirmed,20,20,1.0,0.028169,0.001803
5,r_on_today,confirmed,20,20,1.0,0.027272,0.002321
6,r_cc_mean_5_lag1,confirmed,20,20,1.0,0.026027,0.002637
7,r_on_mean_20,confirmed,20,20,1.0,0.024091,0.001206
8,cc_mom_5_20,confirmed,20,20,1.0,0.021841,0.001572
9,r_on_today_to_vol20,confirmed,20,20,1.0,0.020408,0.002132



Number of confirmed features: 61
Number of tentative features: 14
Number of rejected features: 26

Evaluating 06_core_plus_cross_sectional_rank with 101 features...

Evaluating boruta_06_confirmed with 61 features...

Evaluating boruta_06_confirmed_plus_tentative with 75 features...

Evaluating boruta_06_top30_by_importance with 30 features...

Evaluating boruta_06_top50_by_importance with 50 features...

Evaluating boruta_06_top70_by_importance with 70 features...

Step 3B Boruta validation comparison for 06_core_plus_cross_sectional_rank:


,feature_set,n_features,valid_rank_ic_mean,valid_rank_ic_std,valid_icir_annualized,valid_top_bottom_mean,valid_top_bottom_bps,valid_top_bottom_sharpe,valid_mse,valid_mae,n_ic_days,n_spread_days
0,boruta_06_confirmed_plus_tentative,75,0.029070,0.184896,2.495848,0.000416,4.157610,1.920912,0.00009,0.006789,757,757
1,boruta_06_top70_by_importance,70,0.029083,0.185737,2.485626,0.000407,4.071691,1.870789,0.00009,0.006794,757,757
2,06_core_plus_cross_sectional_rank,101,0.027685,0.178993,2.455331,0.000387,3.874591,1.837727,0.00009,0.006793,757,757
3,boruta_06_confirmed,61,0.028070,0.185168,2.406425,0.000390,3.895901,1.814713,0.00009,0.006789,757,757
4,boruta_06_top50_by_importance,50,0.027090,0.191410,2.246669,0.000378,3.778145,1.733369,0.00009,0.006784,757,757
5,boruta_06_top30_by_importance,30,0.026833,0.185453,2.296906,0.000357,3.565229,1.678200,0.00009,0.006785,757,757



Done.
06 candidate feature list saved to: stage4_feature_selection_outputs\step3b_06_candidate_features.txt
Boruta feature report saved to: stage4_feature_selection_outputs\step3b_boruta_06_feature_report.csv
Validation comparison saved to: stage4_feature_selection_outputs\step3b_boruta_06_validation_comparison.csv
Confirmed features saved to: stage4_feature_selection_outputs\step3b_boruta_06_confirmed_features.txt
Tentative features saved to: stage4_feature_selection_outputs\step3b_boruta_06_tentative_features.txt
Rejected features saved to: stage4_feature_selection_outputs\step3b_boruta_06_rejected_features.txt
Best Step 3B feature set: boruta_06_confirmed_plus_tentative
Best Step 3B feature list saved to: stage4_feature_selection_outputs\step3b_best_boruta_06_feature_set.txt


Bruta 检验结果：
| Boruta 判断   | 数量 | 含义                            |
| ----------- | -: | ----------------------------- |
| `confirmed` | 61 | 明确有用                          |
| `tentative` | 14 | 可能有用，证据不够强，但不能直接删             |
| `rejected`  | 26 | 没有明显超过随机 shadow features，建议删除 |
Boruta 的结果支持你把 101 个高 Sharpe 特征压缩成 75 个 confirmed + tentative 特征，并把这 75 个作为当前最优 feature set

现在从初始数据集中提取这75个特征的所有数据，构成新的数据集:stage4_model_data_boruta75.parquet

In [15]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. 自动寻找 baseline reduced 数据文件
# ============================================================

possible_data_paths = [
    Path("stage4_model_data_baseline_reduced.parquet"),
    Path("stage4_outputs/stage4_model_data_baseline_reduced.parquet"),
    Path("stage4_feature_selection_outputs/stage4_model_data_baseline_reduced.parquet"),
]

data_path = next((p for p in possible_data_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "找不到 stage4_model_data_baseline_reduced.parquet。"
        "请检查文件是否在当前目录、stage4_outputs 或 stage4_feature_selection_outputs 里。"
    )

print("Using data file:", data_path)

df = pd.read_parquet(data_path)
print("Original data shape:", df.shape)


# ============================================================
# 2. 自动寻找 Boruta 75 个特征名单
# ============================================================

possible_feature_paths = [
    Path("stage4_feature_selection_outputs/step3b_boruta_06_confirmed_plus_tentative_features.txt"),
    Path("stage4_feature_selection_outputs/step3b_best_boruta_06_feature_set.txt"),
    Path("step3b_boruta_06_confirmed_plus_tentative_features.txt"),
    Path("step3b_best_boruta_06_feature_set.txt"),
]

feature_path = next((p for p in possible_feature_paths if p.exists()), None)

if feature_path is None:
    raise FileNotFoundError(
        "找不到 Boruta 75 特征名单文件。"
        "请检查 step3b_boruta_06_confirmed_plus_tentative_features.txt "
        "是否在 stage4_feature_selection_outputs 文件夹里。"
    )

print("Using Boruta feature list:", feature_path)

with open(feature_path, "r", encoding="utf-8") as f:
    boruta75_features = [line.strip() for line in f.readlines() if line.strip()]

# 去重，保持顺序
boruta75_features = list(dict.fromkeys(boruta75_features))

print("Number of Boruta selected features:", len(boruta75_features))


# ============================================================
# 3. 检查这 75 个特征是否都在数据里
# ============================================================

missing_features = [c for c in boruta75_features if c not in df.columns]
existing_features = [c for c in boruta75_features if c in df.columns]

print("Existing Boruta features:", len(existing_features))
print("Missing Boruta features:", len(missing_features))

if len(missing_features) > 0:
    print("\nMissing features:")
    for c in missing_features:
        print(c)
    raise ValueError("有 Boruta 特征不在数据文件里，请检查数据文件和特征名单是否匹配。")


# ============================================================
# 4. 保留必要的非特征列
# ============================================================
# 这些列不进入模型 X，但后面训练、验证、回测需要用。

keep_non_feature_cols = [
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "target_r_on_next",
    "target_r_on_next_winsor",
    "is_trade_eligible",
    "eligibility_reason",
]

keep_non_feature_cols = [c for c in keep_non_feature_cols if c in df.columns]

print("\nNon-feature columns kept:")
for c in keep_non_feature_cols:
    print(c)


# ============================================================
# 5. 生成只包含 metadata + target + 75 个特征的新数据
# ============================================================

final_cols = keep_non_feature_cols + boruta75_features

# 去重，保持顺序
final_cols = list(dict.fromkeys(final_cols))

df_boruta75 = df[final_cols].copy()

print("\nBoruta 75 data shape:", df_boruta75.shape)


# ============================================================
# 6. 保存新数据文件和特征列表
# ============================================================

output_dir = Path("stage4_feature_selection_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_data_path = output_dir / "stage4_model_data_boruta75.parquet"
output_feature_path = output_dir / "boruta75_feature_cols.txt"

df_boruta75.to_parquet(output_data_path, index=False)

with open(output_feature_path, "w", encoding="utf-8") as f:
    for col in boruta75_features:
        f.write(col + "\n")

print("\nDone.")
print("Boruta 75 data saved to:", output_data_path)
print("Boruta 75 feature list saved to:", output_feature_path)

print("\nFinal number of model features:", len(boruta75_features))
print("Final data shape:", df_boruta75.shape)

Using data file: stage4_model_data_baseline_reduced.parquet
Original data shape: (3773971, 110)
Using Boruta feature list: stage4_feature_selection_outputs\step3b_boruta_06_confirmed_plus_tentative_features.txt
Number of Boruta selected features: 75
Existing Boruta features: 75
Missing Boruta features: 0

Non-feature columns kept:
date
ticker
instrument_id
sample_split
target_r_on_next
target_r_on_next_winsor
is_trade_eligible
eligibility_reason

Boruta 75 data shape: (3773971, 83)

Done.
Boruta 75 data saved to: stage4_feature_selection_outputs\stage4_model_data_boruta75.parquet
Boruta 75 feature list saved to: stage4_feature_selection_outputs\boruta75_feature_cols.txt

Final number of model features: 75
Final data shape: (3773971, 83)
